# 12 - Leakage-aware cross-validation folds

Splits all 4,407 training studies into 5 folds, used by notebook 14 for training on
weak labels and evaluating on the 58 expert-labelled studies out-of-fold.

**Grouping.** The CSVs have no patient identifier. Two leaks are controlled:
1. identical (template) reports: studies sharing a normalised report hash stay together;
2. repeat studies of one patient: a one-header-per-study scan reads DICOM `PatientID`.
   IDs are hashed immediately and only counts are printed. An ID shared by more than
   20 studies is treated as an anonymisation placeholder and ignored.

**Inputs:** competition data (train_series DICOM headers); notebook 11 output.
**Outputs:** `folds_v1.csv`, `study_headers_v1.csv` (hashed IDs only).
**Runtime:** CPU, no internet; the header scan reads 4,407 files (a few minutes).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
OUT = Path("/kaggle/working")
TARGET_COLUMNS = [
    "ACL", "MCL",
    "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
SEED = 42
from pathlib import Path


def find_input(name):
    """Locate one attached upstream file by name.

    Only notebook outputs and datasets are searched: walking the competition
    mount would visit millions of DICOM files.
    """
    hits = []
    for base in (Path("/kaggle/input/notebooks"), Path("/kaggle/input/datasets"),
                 Path("/kaggle/input")):
        if base.is_dir():
            for child in base.iterdir():
                if (child.is_dir()
                        and child.name not in ("competitions", "notebooks", "datasets")
                        and not (child / "train_series").exists()):
                    hits += list(child.rglob(name))
    hits = sorted(set(hits))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"expected exactly one attached '{name}', found {len(hits)}: {hits[:5]}. "
            "Attach the upstream notebook output via Add Input."
        )
    return hits[0]

import hashlib
from concurrent.futures import ThreadPoolExecutor

import pydicom
from sklearn.model_selection import StratifiedGroupKFold

N_FOLDS = 5
MAX_STUDIES_PER_PATIENT = 20

weak = pd.read_csv(find_input("weak_labels_v1.csv"), dtype={"StudyInstanceUID": str})
series = pd.read_csv(ROOT / "train_series.csv", dtype=str)
assert len(weak) == 4407 and set(weak["StudyInstanceUID"]) == set(series["StudyInstanceUID"])

## 1. One header per study

In [ ]:
def _hash(value):
    value = str(value or "").strip()
    return hashlib.sha256(value.encode()).hexdigest()[:16] if value else ""


def scan(study):
    first = sorted(series.loc[series["StudyInstanceUID"].eq(study), "SeriesInstanceUID"])[0]
    files = sorted((ROOT / "train_series" / study / first).glob("*.dcm"))
    ds = pydicom.dcmread(files[len(files) // 2], stop_before_pixels=True)
    return {"StudyInstanceUID": study,
            "patient_hash": _hash(getattr(ds, "PatientID", "")),
            "laterality_tag": str(getattr(ds, "Laterality", "") or ""),
            "manufacturer": str(getattr(ds, "Manufacturer", "") or "")}


with ThreadPoolExecutor(8) as pool:
    headers = pd.DataFrame(list(pool.map(scan, weak["StudyInstanceUID"])))
headers.to_csv(OUT / "study_headers_v1.csv", index=False)

per_id = headers.loc[headers["patient_hash"].ne(""), "patient_hash"].value_counts()
print("blank PatientID:", int(headers["patient_hash"].eq("").sum()))
print("distinct PatientIDs:", len(per_id), "| studies per ID:", per_id.value_counts().sort_index().to_dict())
print("Laterality tag:", headers["laterality_tag"].replace("", "<absent>").value_counts().to_dict())
print("Manufacturer:", headers["manufacturer"].replace("", "<absent>").value_counts().to_dict())

## 2. Groups: report-hash groups merged with patient groups (union-find)

In [ ]:
parent = {}


def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    parent[find(a)] = find(b)


usable = set(per_id[per_id.between(2, MAX_STUDIES_PER_PATIENT)].index)
print("patient IDs used for grouping (2..20 studies):", len(usable),
      "| ignored as placeholders (>20):", int((per_id > MAX_STUDIES_PER_PATIENT).sum()))

table = weak[["StudyInstanceUID", "is_gold", "report_hash"]].merge(headers, on="StudyInstanceUID")
for row in table.itertuples(index=False):
    union("s:" + row.StudyInstanceUID, "r:" + row.report_hash)
    if row.patient_hash in usable:
        union("s:" + row.StudyInstanceUID, "p:" + row.patient_hash)
table["group"] = [find("s:" + s) for s in table["StudyInstanceUID"]]
sizes = table["group"].value_counts()
print("groups:", len(sizes), "| largest:", int(sizes.max()), "| multi-study groups:", int((sizes > 1).sum()))

## 3. Folds

Stratified on expert-label presence so each fold holds about 11-12 of the 58 studies.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
table["fold"] = -1
for k, (_, idx) in enumerate(sgkf.split(table, table["is_gold"], groups=table["group"])):
    table.loc[table.index[idx], "fold"] = k

assert table["fold"].between(0, N_FOLDS - 1).all()
assert table.groupby("group")["fold"].nunique().eq(1).all(), "a group crosses folds"
gold_per_fold = table.groupby("fold")["is_gold"].sum()
print("studies per fold:", table["fold"].value_counts().sort_index().to_dict())
print("expert-labelled per fold:", gold_per_fold.to_dict())
assert gold_per_fold.between(8, 16).all()

g = table[table["is_gold"].eq(1)].merge(
    weak[["StudyInstanceUID"] + [t + "__gold" for t in TARGET_COLUMNS]], on="StudyInstanceUID")
coverage = pd.DataFrame({t: g.groupby("fold")[t + "__gold"].sum() for t in TARGET_COLUMNS}).T
print("expert positives per fold (a 0 or a full fold makes that fold-target AUC undefined):")
display(coverage.astype(int))

In [ ]:
table[["StudyInstanceUID", "fold", "group", "is_gold"]].to_csv(OUT / "folds_v1.csv", index=False)
print("saved", OUT / "folds_v1.csv")